# PG-LIF — Phase P0: Single-Neuron Validation
**Project:** PG-LIF (Plateau-Gated Leaky Integrate-and-Fire) · **Phase:** P0 of the Research and Implementation Plan
**Feeds:** manuscript Sections 4 and 7.4 (memory-retention curve vs. Eq. 11, Proposition 2 bound, firing regimes, f–I curves, phase diagrams)

This notebook is self-contained (PyTorch + matplotlib only). All figures, CSVs, and a `summary.json` with PASS/FAIL flags are written to a timestamped subfolder of `MyDrive/PG_LIF/P0_results/`. Run all cells top to bottom; total runtime is a few minutes on CPU.

**Decision gate (from the plan):** theory must match simulation before proceeding to Phase P1.

In [ ]:
# --- Setup: mount Drive and create the results folder ---
import os, json, time
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/PG_LIF'
except Exception:
    BASE = './PG_LIF'   # local fallback when not running in Colab
STAMP = time.strftime('%Y%m%d_%H%M%S')
OUT = os.path.join(BASE, 'P0_results', STAMP)
os.makedirs(OUT, exist_ok=True)
print('Results will be saved to:', OUT)

In [ ]:
import math, torch
import numpy as np
import matplotlib.pyplot as plt
torch.set_default_dtype(torch.float64)   # P0 compares against closed forms; use double precision
torch.manual_seed(0); np.random.seed(0)
SUMMARY = {'phase': 'P0', 'timestamp': STAMP, 'checks': {}}

## PG-LIF reference implementation
Vectorized over a batch of `N` neurons (used below to sweep parameter grids in one pass). Implements Eqs. (6)–(10) of the manuscript. The `SpikeFn` surrogate (fast sigmoid) is included for reuse in Phase P1; P0 itself is forward-only.

In [ ]:
class SpikeFn(torch.autograd.Function):
    """Heaviside forward, fast-sigmoid surrogate backward (for Phase P1 training)."""
    scale = 10.0
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return (x >= 0).to(x.dtype)
    @staticmethod
    def backward(ctx, g):
        (x,) = ctx.saved_tensors
        return g / (1.0 + SpikeFn.scale * x.abs()) ** 2

class PGLIF:
    """PG-LIF neurons, batch of N. State: Vs, Vd, p, a (+ refractory counters)."""
    def __init__(self, N, dt=1.0, tau_m=20., tau_d=20., tau_p=200., tau_a=200.,
                 EL=0., Vr=0., theta0=1.0, theta_d=1.0, beta=0.1, kappa=1.0,
                 P0=1.0, Rm=1.0, Rd=1.0, tref=2.0, tref_p=30.0, device='cpu'):
        self.N, self.dt, self.device = N, dt, device
        self.a_m = math.exp(-dt/tau_m); self.a_d = math.exp(-dt/tau_d)
        self.a_p = math.exp(-dt/tau_p); self.a_a = math.exp(-dt/tau_a)
        self.EL, self.Vr, self.theta0 = EL, Vr, theta0
        self.beta, self.P0, self.Rm, self.Rd = beta, P0, Rm, Rd
        self.tau_m, self.tau_d, self.tau_p, self.tau_a = tau_m, tau_d, tau_p, tau_a
        as_vec = lambda v: (v if torch.is_tensor(v) else torch.full((N,), float(v))).to(device)
        self.kappa = as_vec(kappa); self.theta_d = as_vec(theta_d); self.beta = as_vec(beta)
        self.nref = int(round(tref/dt)); self.nref_p = int(round(tref_p/dt))
        self.reset_state()
    def reset_state(self):
        z = lambda: torch.zeros(self.N, device=self.device)
        self.Vs = z()+self.EL; self.Vd = z()+self.EL; self.p = z(); self.a = z()
        self.ref_s = torch.zeros(self.N, dtype=torch.long, device=self.device)
        self.ref_p = torch.zeros(self.N, dtype=torch.long, device=self.device)
    def step(self, Is, Id):
        self.Vd = self.EL + self.a_d*(self.Vd-self.EL) + (1-self.a_d)*self.Rd*Id
        ed = ((self.Vd >= self.theta_d) & (self.ref_p == 0)).double()
        self.ref_p = torch.clamp(self.ref_p-1, min=0)
        self.ref_p = torch.where(ed.bool(), torch.full_like(self.ref_p, self.nref_p), self.ref_p)
        self.p = self.a_p*self.p + self.P0*ed
        self.Vs = self.EL + self.a_m*(self.Vs-self.EL) + (1-self.a_m)*(self.Rm*Is + self.kappa*self.p)
        s = ((self.Vs >= self.theta0 + self.beta*self.a) & (self.ref_s == 0)).double()
        self.ref_s = torch.clamp(self.ref_s-1, min=0)
        self.ref_s = torch.where(s.bool(), torch.full_like(self.ref_s, self.nref), self.ref_s)
        self.Vs = torch.where(s.bool(), torch.full_like(self.Vs, self.Vr), self.Vs)
        self.a = self.a_a*self.a + s
        return s, ed

def run(neuron, Is_seq, Id_seq):
    """Simulate T steps; Is_seq/Id_seq: (T, N) tensors. Returns dict of stacked traces."""
    Vs, Vd, p, a, s, ed = [], [], [], [], [], []
    for t in range(Is_seq.shape[0]):
        st, et = neuron.step(Is_seq[t], Id_seq[t])
        Vs.append(neuron.Vs.clone()); Vd.append(neuron.Vd.clone())
        p.append(neuron.p.clone()); a.append(neuron.a.clone()); s.append(st); ed.append(et)
    return {k: torch.stack(v) for k, v in
            dict(Vs=Vs, Vd=Vd, p=p, a=a, s=s, ed=ed).items()}

## Experiment 1 — Memory trace of a single dendritic event vs. Eq. (11)
One suprathreshold dendritic pulse at t = 0, spiking disabled, no further input. The measured somatic depolarization must match the closed form of Eq. (11) to numerical precision, and the retention time must be governed by τp.

In [ ]:
T = 800
n = PGLIF(1, theta0=1e9)   # disable somatic spiking to observe the subthreshold trace
Is = torch.zeros(T, 1); Id = torch.zeros(T, 1); Id[0, 0] = 25.0   # (1-a_d)*Rd*Id >= theta_d
tr = run(n, Is, Id)
sim = (tr['Vs'][:, 0] - n.EL).numpy()
am, ap, k, P0 = n.a_m, n.a_p, float(n.kappa[0]), n.P0
theory = np.array([k*P0*(1-am)*(ap**(t+1)-am**(t+1))/(ap-am) for t in range(T)])
err = float(np.abs(sim - theory).max())
ok1 = err < 1e-9
SUMMARY['checks']['eq11_match'] = {'max_abs_err': err, 'pass': ok1}
print(f'max |sim - Eq.(11)| = {err:.2e}  ->', 'PASS' if ok1 else 'FAIL')

plt.figure(figsize=(7, 4))
plt.plot(sim, lw=3, label='simulation')
plt.plot(theory, 'k--', lw=1, label='Eq. (11) closed form')
plt.xlabel('time after dendritic event (ms)'); plt.ylabel(r'$\Delta V_s$')
plt.title(f'Single-event memory trace (tau_p = {n.tau_p:.0f} ms); max err {err:.1e}')
plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig_E1_memory_trace.png'), dpi=300)
np.savetxt(os.path.join(OUT, 'E1_memory_trace.csv'),
           np.column_stack([np.arange(T), sim, theory]),
           delimiter=',', header='t_ms,sim_dVs,theory_dVs', comments='')
plt.show()

## Experiment 2 — Proposition 2: boundedness of the plateau state
Constant strong dendritic drive (worst case: an event every refractory period). The plateau state must stay below P0 / (1 − αp^nref).

In [ ]:
T = 5000
n = PGLIF(1, theta0=1e9)
tr = run(n, torch.zeros(T, 1), torch.full((T, 1), 5.0))
pmax = float(tr['p'].max()); bound = n.P0 / (1 - n.a_p ** n.nref_p)
ok2 = pmax <= bound + 1e-9
SUMMARY['checks']['prop2_bound'] = {'p_max': pmax, 'bound': bound, 'pass': ok2}
print(f'max p = {pmax:.4f} <= bound {bound:.4f} ->', 'PASS' if ok2 else 'FAIL')

plt.figure(figsize=(7, 3.2))
plt.plot(tr['p'][:, 0].numpy(), lw=1)
plt.axhline(bound, color='r', ls='--', label='Prop. 2 bound')
plt.xlabel('time (ms)'); plt.ylabel('p(t)'); plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig_E2_plateau_bound.png'), dpi=300); plt.show()

## Experiment 3 — Firing regimes
Three configurations demonstrate the regimes of Section 4.3: **tonic** (κ = 0, suprathreshold soma drive), **plateau-driven** (soma drive subthreshold on its own — control must be silent — firing only via dendritic events), and **adaptive bursting** (large β terminates plateau-driven firing episodes).

In [ ]:
T = 1500
def Id_pulses(times, amp=25.0):
    x = torch.zeros(T, 1)
    for t0 in times: x[t0, 0] = amp
    return x
cfgs = {
 'tonic':          dict(kappa=0.0, beta=0.0, Is=1.5, Id=torch.zeros(T, 1)),
 'soma-alone ctrl':dict(kappa=0.0, beta=0.0, Is=0.6, Id=torch.zeros(T, 1)),
 'plateau-driven': dict(kappa=2.0, beta=0.0, Is=0.6, Id=Id_pulses([100, 600, 1100])),
 'adaptive burst': dict(kappa=2.0, beta=1.5, Is=0.6, Id=Id_pulses([100, 600, 1100])),
}
traces, counts = {}, {}
for name, cfg in cfgs.items():
    n = PGLIF(1, kappa=cfg['kappa'], beta=cfg['beta'])
    tr = run(n, torch.full((T, 1), cfg['Is']), cfg['Id'])
    traces[name] = tr; counts[name] = int(tr['s'].sum())
ok3 = counts['soma-alone ctrl'] == 0 and counts['plateau-driven'] > 0 \
      and counts['adaptive burst'] < counts['plateau-driven'] and counts['tonic'] > 0
SUMMARY['checks']['regimes'] = {'spike_counts': counts, 'pass': ok3}
print(counts, '->', 'PASS' if ok3 else 'FAIL')

fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=True)
for ax, name in zip(axes, ['tonic', 'plateau-driven', 'adaptive burst']):
    tr = traces[name]
    ax.plot(tr['Vs'][:, 0].numpy(), lw=0.8, label='Vs')
    ax.plot(tr['p'][:, 0].numpy(), lw=1.2, label='p')
    st = torch.nonzero(tr['s'][:, 0]).flatten().numpy()
    ymax = float(tr['Vs'].max())
    ax.vlines(st, ymax * 1.02, ymax * 1.15, color='k', lw=0.5)
    et = torch.nonzero(tr['ed'][:, 0]).flatten().numpy()
    if len(et): ax.plot(et, [0.0] * len(et), 'r^', ms=6, label='dendritic event')
    ax.set_ylabel(name, fontsize=9); ax.legend(fontsize=7, loc='upper right')
axes[-1].set_xlabel('time (ms)')
plt.suptitle('PG-LIF firing regimes (spike ticks above; red triangles: dendritic events)')
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig_E3_regimes.png'), dpi=300); plt.show()

## Experiment 4 — f–I curves
Firing rate vs. somatic input, with the dendritic pathway off (κ = 0) and on (κ = 2 with background dendritic drive). The plateau pathway should shift the curve leftward: weaker somatic inputs suffice after dendritic conjunctions — the excitability trace of Experiment 1 at network-relevant drive levels. The sweep is vectorized across the batch dimension.

In [ ]:
T = 3000; Is_grid = torch.linspace(0.0, 3.0, 31)
rates = {}
for label, (kap, Idrive) in {'kappa=0': (0.0, 0.0), 'kappa=2, Id=1.2': (2.0, 1.2)}.items():
    n = PGLIF(len(Is_grid), kappa=kap)
    tr = run(n, Is_grid.repeat(T, 1), torch.full((T, len(Is_grid)), Idrive))
    rates[label] = (tr['s'].sum(0) / (T / 1000.0)).numpy()   # Hz
plt.figure(figsize=(6.5, 4))
for label, r in rates.items(): plt.plot(Is_grid.numpy(), r, 'o-', ms=3, label=label)
plt.xlabel('somatic input Is'); plt.ylabel('firing rate (Hz)'); plt.legend(); plt.tight_layout()
plt.title('f-I curves with and without the plateau pathway')
plt.savefig(os.path.join(OUT, 'fig_E4_fI.png'), dpi=300)
np.savetxt(os.path.join(OUT, 'E4_fI.csv'),
           np.column_stack([Is_grid.numpy()] + list(rates.values())), delimiter=',',
           header='Is,' + ','.join(rates.keys()), comments='')
plt.show()

## Experiment 5 — Phase diagram over (κ, β) and plateau activation map
Left: regime classification on a κ×β grid at fixed drives (soma subthreshold alone), classified by spike count and ISI irregularity. Right: does a dendritic pulse of given amplitude and duration trigger a plateau event (activation boundary of Eq. 2)? Both grids run as one vectorized batch.

In [ ]:
# --- phase diagram ---
T = 2000; K = 15
kaps = torch.linspace(0.0, 3.0, K); betas = torch.linspace(0.0, 2.0, K)
KK, BB = torch.meshgrid(kaps, betas, indexing='ij')
n = PGLIF(K*K, kappa=KK.flatten(), beta=BB.flatten())
tr = run(n, torch.full((T, K*K), 0.6), torch.full((T, K*K), 1.5))
counts = tr['s'].sum(0)
cls = torch.zeros(K*K)                     # 0 silent
for i in range(K*K):
    st = torch.nonzero(tr['s'][:, i]).flatten().double()
    if len(st) < 3: cls[i] = 0 if len(st) == 0 else 1; continue
    isi = st[1:] - st[:-1]
    cv = (isi.std() / isi.mean()).item()
    cls[i] = 3 if cv > 0.8 else 2          # 3 bursting, 2 regular
SUMMARY['checks']['phase_diagram'] = {'n_silent': int((cls==0).sum()), 'n_sparse': int((cls==1).sum()),
                                      'n_regular': int((cls==2).sum()), 'n_bursting': int((cls==3).sum())}
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
im = axes[0].imshow(cls.reshape(K, K).T.numpy(), origin='lower', aspect='auto', cmap='viridis',
                    extent=[kaps[0], kaps[-1], betas[0], betas[-1]])
axes[0].set_xlabel('kappa'); axes[0].set_ylabel('beta')
axes[0].set_title('Regimes (0 silent, 1 sparse, 2 regular, 3 bursting)')
plt.colorbar(im, ax=axes[0])

# --- plateau activation map ---
amps = torch.linspace(0.5, 25.0, 25); durs = torch.arange(1, 26).double()
AA, DD = torch.meshgrid(amps, durs, indexing='ij')
n2 = PGLIF(len(amps)*len(durs), theta0=1e9)
Tp = 60
Id_seq = torch.zeros(Tp, len(amps)*len(durs))
for t in range(Tp):
    Id_seq[t] = torch.where(torch.tensor(t) < DD.flatten(), AA.flatten(), torch.zeros_like(AA.flatten()))
tr2 = run(n2, torch.zeros(Tp, len(amps)*len(durs)), Id_seq)
fired = (tr2['ed'].sum(0) > 0).reshape(len(amps), len(durs))
axes[1].imshow(fired.T.numpy(), origin='lower', aspect='auto', cmap='Greys',
               extent=[amps[0], amps[-1], durs[0].item(), durs[-1].item()])
axes[1].set_xlabel('dendritic pulse amplitude'); axes[1].set_ylabel('pulse duration (ms)')
axes[1].set_title('Plateau activation map (black = event)')
plt.tight_layout(); plt.savefig(os.path.join(OUT, 'fig_E5_phase_activation.png'), dpi=300); plt.show()

## Summary and decision gate

In [ ]:
SUMMARY['all_pass'] = all(v.get('pass', True) for v in SUMMARY['checks'].values())
with open(os.path.join(OUT, 'summary.json'), 'w') as f:
    json.dump(SUMMARY, f, indent=2)
print(json.dumps(SUMMARY, indent=2))
print()
print('Saved to:', OUT)
print('DECISION GATE P0:', 'PASSED - proceed to Phase P1 (Regime A on SHD)' if SUMMARY['all_pass']
      else 'FAILED - fix implementation/theory before P1')

### Next steps
1. Insert `fig_E1`, `fig_E3`, `fig_E4`, `fig_E5` into manuscript Section 7.4 and replace the corresponding red `[[RESULT]]` markers, describing what was actually observed.
2. Phase P1: reuse the `PGLIF` class and `SpikeFn` above inside an `nn.Module` recurrent layer; benchmark on SHD against LIF/ALIF/TC-LIF/DH-LIF (plan Section 5, Table 2).
3. Phase P2 (before SSC/PS-MNIST): ablation (a) — replace the plateau with a second ALIF variable of equal time constant.